# Part B: Neural network regression on the Runge function

Evaluates the FFNN and compares it with the Project 1 OLS baseline.

## Data and preprocessing

Use the same Runge data-generation parameters as the existing Project 2 setup. The original 20-point test split remains untouched; only the 80 training points are split into training and validation sets. Scaling statistics are fitted on the training subset only.

In [ ]:
import json
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LogNorm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from data import make_runge_data, sigma
from nn import NeuralNetwork, train
from optimizers import Adam, GD, RMSprop

jax.config.update("jax_enable_x64", True)

X_train_all, X_test_original, y_train_all, y_test_original = make_runge_data(
    100, sigma, 0.2, 2026
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all,
    y_train_all,
    test_size=0.25,
    random_state=2026,
)

# Fit input scaling and target centering on the training subset only.
x_mean = X_train.mean(axis=0)
x_std = X_train.std(axis=0)
X_train_scaled = (X_train - x_mean) / x_std
X_val_scaled = (X_val - x_mean) / x_std
y_mean = y_train.mean()
y_train_centered = y_train - y_mean
y_val_centered = y_val - y_mean

print(f"Train: {len(y_train)}, validation: {len(y_val)}, test: {len(y_test_original)}")
print(f"Noise standard deviation sigma={sigma}; noise floor sigma^2={sigma**2:.6f}")

Train: 60, validation: 20, test: 20
Noise standard deviation sigma=0.1; noise floor sigma^2=0.010000


## Gradient check

Compare the manual backpropagation gradients with JAX automatic differentiation for a [1, 5, 5, 1] sigmoid/sigmoid/identity network and ten data points.

In [2]:
gradient_model = NeuralNetwork(
    [1, 5, 5, 1], ["sigmoid", "sigmoid", "identity"], cost="mse", seed=2026
)
X_check = X_train_scaled[:10]
y_check = y_train_centered[:10]
manual_grads = gradient_model.backward(X_check, y_check)
params_jax = [(jnp.asarray(W), jnp.asarray(b)) for W, b in gradient_model.params]
X_check_jax = jnp.asarray(X_check)
y_check_jax = jnp.asarray(y_check)

def jax_sigmoid_forward_mse(params):
    activation = X_check_jax
    for layer_index, (W, b) in enumerate(params):
        z = activation @ W + b
        if layer_index < len(params) - 1:
            activation = jax.nn.sigmoid(z)
        else:
            activation = z
    return jnp.sum((activation - y_check_jax) ** 2) / y_check_jax.shape[0]

jax_grads = jax.grad(jax_sigmoid_forward_mse)(params_jax)
for layer_index, ((manual_dW, manual_db), (jax_dW, jax_db)) in enumerate(
    zip(manual_grads, jax_grads), start=1
):
    jax_dW_np = np.asarray(jax_dW)
    jax_db_np = np.asarray(jax_db)
    print(
        f"Layer {layer_index} dW: allclose={np.allclose(manual_dW, jax_dW_np, rtol=1e-7, atol=1e-9)}, "
        f"max_abs_diff={np.max(np.abs(manual_dW - jax_dW_np)):.3e}"
    )
    print(
        f"Layer {layer_index} db: allclose={np.allclose(manual_db, jax_db_np, rtol=1e-7, atol=1e-9)}, "
        f"max_abs_diff={np.max(np.abs(manual_db - jax_db_np)):.3e}"
    )

Layer 1 dW: allclose=True, max_abs_diff=4.337e-19
Layer 1 db: allclose=True, max_abs_diff=4.337e-19
Layer 2 dW: allclose=True, max_abs_diff=1.084e-18
Layer 2 db: allclose=True, max_abs_diff=8.674e-19
Layer 3 dW: allclose=True, max_abs_diff=1.908e-17
Layer 3 db: allclose=True, max_abs_diff=1.041e-17


## Learning-rate and architecture experiments

This is a coarse logarithmic sweep: nine values span four orders of magnitude from 1e-4 to 1, with a constant ratio of about 3.16 between neighbors. The assignment asks for different learning rates but does not prescribe nine; this grid is an exploratory choice. Training uses a fixed budget of 1000 epochs, not an early-stopping convergence test. In this experiment GD uses full batches while RMSprop and Adam use batches of 16, so optimizer comparisons also reflect a different batch/update schedule.

In [ ]:
load_saved_results = False
architectures = [[1, 50, 1], [1, 100, 1], [1, 50, 50, 1], [1, 100, 100, 1]]
optimizers_name = ["GD", "RMSprop", "Adam"]
learning_rates = np.logspace(-4, 0, 9)
epochs = 1000
seeds = [2026, 2027, 2028, 2029, 2030]
batch_sizes = {"GD": None, "RMSprop": 16, "Adam": 16}

results_dir = Path("../results")
experiment_csv = results_dir / "part_b_experiments.csv"
histories_npz = results_dir / "part_b_histories.npz"


def make_optimizer(name, learning_rate):
    if name == "GD":
        return GD(learning_rate=learning_rate)
    if name == "RMSprop":
        return RMSprop(learning_rate=learning_rate)
    if name == "Adam":
        return Adam(learning_rate=learning_rate)
    raise ValueError(f"Unknown optimizer: {name}")


if load_saved_results:
    if not experiment_csv.exists() or not histories_npz.exists():
        raise FileNotFoundError("Saved experiment CSV/NPZ not found; set load_saved_results=False first.")
    results = pd.read_csv(experiment_csv)
    histories = dict(np.load(histories_npz))
else:
    records = []
    histories = {}
    run_number = 0
    for architecture in architectures:
        activation_names = ["sigmoid"] * (len(architecture) - 2) + ["identity"]
        for optimizer_name in optimizers_name:
            for learning_rate in learning_rates:
                for seed in seeds:
                    run_key = f"run_{run_number:04d}"
                    model = NeuralNetwork(
                        architecture, activation_names, cost="mse", l1=0.0, l2=0.0, seed=seed
                    )
                    optimizer = make_optimizer(optimizer_name, float(learning_rate))
                    history = train(
                        model,
                        optimizer,
                        X_train_scaled,
                        y_train_centered,
                        epochs,
                        batch_size=batch_sizes[optimizer_name],
                        seed=seed,
                        X_val=X_val_scaled,
                        y_val=y_val_centered,
                        stop_on_nonfinite=True,
                    )
                    histories[run_key] = np.column_stack((history["train"], history["val"]))
                    final_train_mse = history["train"][-1]
                    final_validation_mse = history["val"][-1]
                    records.append({
                        "architecture": json.dumps(architecture),
                        "optimizer": optimizer_name,
                        "learning_rate": float(learning_rate),
                        "seed": seed,
                        "final_train_mse": final_train_mse,
                        "final_validation_mse": final_validation_mse,
                        "history_key": run_key,
                    })
                    run_number += 1
    results = pd.DataFrame(records)
    results.to_csv(experiment_csv, index=False)
    np.savez_compressed(histories_npz, **histories)
    print(f"Saved {len(results)} runs to {experiment_csv} and {histories_npz}")

print(f"Loaded/created {len(results)} experiment records.")
results.head()

## Final evaluation

For each architecture, select the optimizer and learning rate by mean validation MSE across all five seeds. Refit scaling on all 80 training observations, retrain, and score the untouched test set on the original target scale.

In [ ]:
# Keep every finite validation score, including large errors; only non-finite runs are unavailable.
selection_results = results.replace([np.inf, -np.inf], np.nan).copy()

group_columns = ["architecture", "optimizer", "learning_rate"]
selection = (
    selection_results.groupby(group_columns, as_index=False)["final_validation_mse"]
    .agg(
        mean_validation_mse="mean",
        std_validation_mse="std",
        valid_seed_count="count",
    )
)
selection.loc[selection["valid_seed_count"] != len(seeds), "mean_validation_mse"] = np.nan
valid_selection = selection[np.isfinite(selection["mean_validation_mse"])]
if valid_selection.empty:
    raise RuntimeError("No configuration has finite validation MSE for all five seeds.")

# Refit scaling using all 80 training points; do not use test data for fitting.
x_mean_final = X_train_all.mean(axis=0)
x_std_final = X_train_all.std(axis=0)
X_train_all_scaled = (X_train_all - x_mean_final) / x_std_final
X_test_final_scaled = (X_test_original - x_mean_final) / x_std_final
y_mean_final = y_train_all.mean()
y_train_all_centered = y_train_all - y_mean_final

selected_configurations = []
network_test_predictions = {}
for architecture in architectures:
    architecture_key = json.dumps(architecture)
    architecture_selection = valid_selection[
        valid_selection["architecture"] == architecture_key
    ]
    if architecture_selection.empty:
        raise RuntimeError(
            f"No configuration for architecture {architecture} has five finite seeds."
        )
    best_configuration = architecture_selection.loc[
        architecture_selection["mean_validation_mse"].idxmin()
    ]
    optimizer_name = best_configuration["optimizer"]
    learning_rate = float(best_configuration["learning_rate"])
    activations = ["sigmoid"] * (len(architecture) - 2) + ["identity"]
    final_model = NeuralNetwork(
        architecture, activations, cost="mse", l1=0.0, l2=0.0, seed=seeds[0]
    )
    final_optimizer = make_optimizer(optimizer_name, learning_rate)
    final_history = train(
        final_model,
        final_optimizer,
        X_train_all_scaled,
        y_train_all_centered,
        epochs,
        batch_size=batch_sizes[optimizer_name],
        seed=seeds[0],
        stop_on_nonfinite=True,
    )
    if not np.isfinite(final_history["train"][-1]):
        raise RuntimeError(
            f"Final retraining became non-finite for architecture {architecture}."
        )
    predictions = final_model.predict(X_test_final_scaled) + y_mean_final
    network_test_predictions[architecture_key] = predictions
    selected_configurations.append({
        "architecture": architecture_key,
        "optimizer": optimizer_name,
        "learning_rate": learning_rate,
        "mean_validation_mse": float(best_configuration["mean_validation_mse"]),
        "std_validation_mse": float(best_configuration["std_validation_mse"]),
        "test_mse": float(np.mean((predictions - y_test_original) ** 2)),
    })

architecture_results = pd.DataFrame(selected_configurations)
best_architecture_result = architecture_results.loc[
    architecture_results["mean_validation_mse"].idxmin()
]
best_architecture = json.loads(best_architecture_result["architecture"])
best_network_test_predictions = network_test_predictions[
    best_architecture_result["architecture"]
]
print("Selected and retrained configuration for each architecture:")
print(architecture_results.to_string(index=False))

## OLS baseline

This follows Project 1: use an increasing-power Vandermonde design matrix, standardize polynomial columns but leave the intercept column unchanged, fit on all 80 training points, and evaluate on the same untouched test set. Fill in the best degree from Project 1 before running this section.

In [ ]:
DEGREE = 8
ols_test_mse = np.nan
X_ols_train = np.vander(X_train_all.ravel(), N=DEGREE + 1, increasing=True)
X_ols_test = np.vander(X_test_original.ravel(), N=DEGREE + 1, increasing=True)
polynomial_scaler = StandardScaler()
X_ols_train_scaled = X_ols_train.copy()
X_ols_test_scaled = X_ols_test.copy()
if DEGREE > 0:
    X_ols_train_scaled[:, 1:] = polynomial_scaler.fit_transform(X_ols_train[:, 1:])
    X_ols_test_scaled[:, 1:] = polynomial_scaler.transform(X_ols_test[:, 1:])
ols_coefficients = np.linalg.lstsq(X_ols_train_scaled, y_train_all, rcond=None)[0]
y_ols_test = X_ols_test_scaled @ ols_coefficients
ols_test_mse = float(np.mean((y_ols_test - y_test_original) ** 2))
print(f"OLS test MSE (original y scale, degree={DEGREE}): {ols_test_mse:.8f}")

network_table = architecture_results.copy()
network_table["Architecture"] = network_table["architecture"].map(
    lambda value: str(json.loads(value))
)
network_table["Mean +/- std validation MSE"] = network_table.apply(
    lambda row: f"{row['mean_validation_mse']:.6g} +/- {row['std_validation_mse']:.3g}",
    axis=1,
)
network_table = network_table.rename(columns={
    "optimizer": "Optimizer",
    "learning_rate": "Learning rate",
    "test_mse": "Test MSE (original scale)",
})
network_table = network_table[[
    "Architecture", "Optimizer", "Learning rate",
    "Mean +/- std validation MSE", "Test MSE (original scale)",
]]
ols_row = pd.DataFrame([{
    "Architecture": f"OLS (degree {DEGREE})" if DEGREE is not None else "OLS",
    "Optimizer": "OLS",
    "Learning rate": np.nan,
    "Mean +/- std validation MSE": "N/A",
    "Test MSE (original scale)": ols_test_mse,
}])
comparison_table = pd.concat([network_table, ols_row], ignore_index=True)
print(comparison_table.to_string(index=False))

if y_ols_test is not None:
    best_squared_errors = (best_network_test_predictions - y_test_original).ravel() ** 2
    ols_squared_errors = (y_ols_test - y_test_original).ravel() ** 2
    squared_error_difference = best_squared_errors - ols_squared_errors
    difference_mean = float(np.mean(squared_error_difference))
    difference_standard_error = float(
        np.std(squared_error_difference, ddof=1) / np.sqrt(len(squared_error_difference))
    )
    print(
        "Best network minus OLS per-point squared-error difference: "
        f"mean={difference_mean:.8f}, standard error={difference_standard_error:.8f}"
    )
comparison_table

## Plots

Heatmaps use the mean of available finite validation scores for each configuration; only configurations with no finite scores are blank. Non-finite runs are counted separately. All finite scores, including values above 1, are included. Colors are capped at the 95th percentile to preserve contrast, while annotations show the actual means. Learning curves use the best mean-validation configuration for each optimizer, averaged over five seeds, with min-max bands.

In [ ]:
# Keep all finite values in the heatmaps and report large/non-finite results separately.
raw_validation_mse = results["final_validation_mse"].to_numpy(dtype=float)
nonfinite_runs = ~np.isfinite(raw_validation_mse)
large_error_runs = np.isfinite(raw_validation_mse) & (raw_validation_mse > 1)
results = results.replace([np.inf, -np.inf], np.nan)
learning_rate_labels = [f"{rate:.0e}" for rate in learning_rates]
results["learning_rate_label"] = results["learning_rate"].map(
    lambda rate: f"{rate:.0e}"
)

def count_by_optimizer_and_rate(mask):
    return (
        results.assign(flag=mask)
        .groupby(["optimizer", "learning_rate_label"])["flag"]
        .sum()
        .unstack("learning_rate_label", fill_value=0)
        .reindex(index=optimizers_name, columns=learning_rate_labels)
        .fillna(0)
        .astype(int)
    )

large_error_counts = count_by_optimizer_and_rate(large_error_runs)
nonfinite_counts = count_by_optimizer_and_rate(nonfinite_runs)
print("Finite runs with validation MSE > 1 (included in heatmaps):")
print(large_error_counts.to_string())
print("Non-finite runs (cannot be represented on a log heatmap):")
print(nonfinite_counts.to_string())

heatmap_results = results.copy()
heatmap_matrices = []
for architecture in architectures:
    architecture_key = json.dumps(architecture)
    subset = heatmap_results[heatmap_results["architecture"] == architecture_key]
    matrix = (
        subset.groupby(["optimizer", "learning_rate_label"])["final_validation_mse"]
        .mean()
        .unstack("learning_rate_label")
        .reindex(index=optimizers_name, columns=learning_rate_labels)
    )
    heatmap_matrices.append(matrix)

all_values = np.concatenate([
    matrix.to_numpy(dtype=float).ravel() for matrix in heatmap_matrices
])
finite_positive = all_values[np.isfinite(all_values) & (all_values > 0)]
if finite_positive.size == 0:
    raise RuntimeError("No finite positive validation MSE values are available to plot.")
vmin = finite_positive.min()
color_vmax = float(np.percentile(finite_positive, 95))
if color_vmax <= vmin:
    color_vmax = vmin * (1.0 + 1e-6)
norm = LogNorm(vmin=vmin, vmax=color_vmax, clip=False)
cmap = plt.get_cmap("viridis").copy()
cmap.set_over("#8b0000")
fig, axes = plt.subplots(2, 2, figsize=(19, 11), constrained_layout=False)
for architecture, matrix, ax in zip(architectures, heatmap_matrices, axes.flat):
    sns.heatmap(
        matrix,
        mask=matrix.isna(),
        xticklabels=learning_rate_labels,
        yticklabels=optimizers_name,
        norm=norm,
        cmap=cmap,
        annot=True,
        fmt=".2g",
        cbar=False,
        ax=ax,
    )
    ax.set_xlabel("Learning rate")
    ax.set_ylabel("Optimizer")
    ax.set_title(f"Architecture {architecture}")
    ax.tick_params(axis="x", labelrotation=0)
    ax.tick_params(axis="y", labelrotation=0)
fig.subplots_adjust(right=0.90, wspace=0.30, hspace=0.32)
colorbar_ax = fig.add_axes([0.92, 0.18, 0.018, 0.64])
colorbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=colorbar_ax,
    extend="max",
)
colorbar.set_label("Mean validation MSE (log color; values above 95th percentile saturated)")
figures_dir = Path("../figures")
figures_dir.mkdir(parents=True, exist_ok=True)
heatmap_path = figures_dir / "part_b_heatmaps.pdf"
fig.savefig(heatmap_path, bbox_inches="tight")
plt.show()

# Choose each optimizer's best architecture/rate, then summarize its five histories.
curve_summaries = {}
for optimizer_name in optimizers_name:
    candidates = valid_selection[valid_selection["optimizer"] == optimizer_name]
    if candidates.empty:
        raise RuntimeError(f"No five-seed configuration is available for {optimizer_name}.")
    best_configuration = candidates.loc[candidates["mean_validation_mse"].idxmin()]
    matching_runs = selection_results[
        (selection_results["architecture"] == best_configuration["architecture"])
        & (selection_results["optimizer"] == optimizer_name)
        & (selection_results["learning_rate"] == best_configuration["learning_rate"])
    ].sort_values("seed")
    if len(matching_runs) != len(seeds):
        raise RuntimeError(f"Expected five seed histories for {optimizer_name}.")
    run_histories = [histories[key] for key in matching_runs["history_key"]]
    max_epochs = max(len(history) for history in run_histories)
    aligned = np.full((len(run_histories), max_epochs, 2), np.nan)
    for run_index, history in enumerate(run_histories):
        aligned[run_index, :len(history), :] = history

    summary = {}
    for metric_index, metric_name in enumerate(("train", "validation")):
        metric_histories = aligned[:, :, metric_index]
        count = np.sum(np.isfinite(metric_histories), axis=0)
        mean_curve = np.divide(
            np.nansum(metric_histories, axis=0),
            count,
            out=np.full(max_epochs, np.nan),
            where=count > 0,
        )
        min_curve = np.full(max_epochs, np.nan)
        max_curve = np.full(max_epochs, np.nan)
        for epoch_index in np.flatnonzero(count):
            values = metric_histories[:, epoch_index]
            values = values[np.isfinite(values)]
            min_curve[epoch_index] = values.min()
            max_curve[epoch_index] = values.max()
        summary[metric_name] = (mean_curve, min_curve, max_curve)

    curve_summaries[optimizer_name] = {
        "epochs": np.arange(1, max_epochs + 1),
        "architecture": json.loads(best_configuration["architecture"]),
        "learning_rate": float(best_configuration["learning_rate"]),
        "curves": summary,
    }

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)
line_colors = {"train": "tab:blue", "validation": "tab:orange"}
for optimizer_name, ax in zip(optimizers_name, axes):
    summary = curve_summaries[optimizer_name]
    for metric_name in ("train", "validation"):
        mean_curve, min_curve, max_curve = summary["curves"][metric_name]
        color = line_colors[metric_name]
        linestyle = "-" if metric_name == "train" else "--"
        ax.plot(
            summary["epochs"], mean_curve, color=color, linestyle=linestyle,
            label=f"Mean {metric_name}",
        )
        ax.fill_between(
            summary["epochs"], min_curve, max_curve, color=color, alpha=0.16,
            label=f"Min-max {metric_name}",
        )
    ax.axhline(sigma**2, color="black", linestyle=":", label=r"Noise floor $\sigma^2$")
    ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_title(
        f"{optimizer_name}\n{summary['architecture']}, "
        f"lr={summary['learning_rate']:.1e}"
    )
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(fontsize="x-small")
axes[0].set_ylabel("MSE")
fig.suptitle("Mean learning curves and min-max range over five seeds")
fig.tight_layout()
learning_curves_path = figures_dir / "part_b_learning_curves.pdf"
fig.savefig(learning_curves_path, bbox_inches="tight")
plt.show()

# Save individual panels too, so they can be combined as LaTeX subfigures if desired.
for optimizer_name in optimizers_name:
    summary = curve_summaries[optimizer_name]
    single_fig, single_ax = plt.subplots(figsize=(6, 4.5))
    for metric_name in ("train", "validation"):
        mean_curve, min_curve, max_curve = summary["curves"][metric_name]
        color = line_colors[metric_name]
        linestyle = "-" if metric_name == "train" else "--"
        single_ax.plot(
            summary["epochs"], mean_curve, color=color, linestyle=linestyle,
            label=f"Mean {metric_name}",
        )
        single_ax.fill_between(
            summary["epochs"], min_curve, max_curve, color=color, alpha=0.16,
            label=f"Min-max {metric_name}",
        )
    single_ax.axhline(sigma**2, color="black", linestyle=":", label=r"Noise floor $\sigma^2$")
    single_ax.set_yscale("log")
    single_ax.set_xlabel("Epoch")
    single_ax.set_ylabel("MSE")
    single_ax.set_title(
        f"{optimizer_name}: {summary['architecture']}, "
        f"lr={summary['learning_rate']:.1e}"
    )
    single_ax.grid(True, which="both", alpha=0.25)
    single_ax.legend(fontsize="small")
    single_fig.tight_layout()
    single_path = figures_dir / f"part_b_learning_curve_{optimizer_name}.pdf"
    single_fig.savefig(single_path, bbox_inches="tight")
    plt.close(single_fig)

print(f"Saved full heatmaps to {heatmap_path}")
print(f"Saved three-panel curves to {learning_curves_path}")
print("Saved individual learning-curve panels for GD, RMSprop, and Adam.")